In [1]:
using LensFactory
using LensFactory.Constants
using LensFactory.LensModel.LensModelIO
using JLD2
using Interpolations
using CairoMakie
using FITSIO
using ProgressMeter

include("FreeFormLens.jl")

Main.FreeFormLens

In [ ]:
function make_gridfrom_model(model::LensModel.ModelConfig)
    """
    Constructs a grid based on the model configuration.
    Returns the grid coordinates (gridx, gridy).
    """
    X_max, Y_max = model.observation.FOV
    pixel_scale = model.observation.pixel_scale
    gridx, gridy = Lenses.get_meshgrid(X_max, Y_max, pixel_scale)

    return gridx, gridy
end

function refine_map(map_coarse::M, gridx::M, gridy::M, new_grid_limx::T, new_grid_limy::T, resolution::U) where {T <:RV, U <:RV, M <: ROA}
    """
    Uses Bicubic interpolation to return a higher res map of the convergence κ.
    """
    x_nodes = range(gridx[1, 1], stop=gridx[end, 1], length=size(gridx, 2))
    y_nodes = range(gridy[1, 1], stop=gridy[1, end], length=size(gridy, 1))

    itp = interpolate(map_coarse, BSpline(Cubic(Line(OnGrid()))))
    itp = scale(itp, x_nodes, y_nodes)

    x_fine, y_fine = Lenses.get_meshgrid(new_grid_limx, new_grid_limy, resolution)

    map_fine = itp.(x_fine, y_fine)


    println("before returning")
    println(size(map_fine))

    return map_fine, x_fine, y_fine

end

function predict_image(lens::Lenses.AbstractLens, gridx::M, gridy::M, θx::N, θy::N, adis::T, sid::Int, kid::Int, images_obs, plot_flag::Bool, path::String) where {T <: RV, M <: ROA, N <: ROA}
    """
    Predicts the image positions based on the lens model and source positions.
    """
    αx, αy = Lenses.get_deflection(lens, θx, θy)

    βx = θx .- αx .* adis
    βy = θy .- αy .* adis

    μ_obs = Lenses.get_magnification_image(lens, θx, θy, adis)

    # Calculate barycenter source position
    βx_model = sum(βx .* μ_obs.^2) / sum(μ_obs.^2)
    βy_model = sum(βy .* μ_obs.^2) / sum(μ_obs.^2)

    images = Lenses.get_image(lens, gridx, gridy, adis, (βx_model, βy_model))

    if plot_flag
        fig, axes = Lenses.plot_image_plane(lens, gridx, gridy, adis, source=(βx_model, βy_model), two_panel=true)
        makie_points = [Point2f(pt) for pt in images_obs]
        scatter!(axes[2], makie_points, color=:cyan, markersize=8)
        outpath = path * "images_sid$(sid)_kid$(kid).png"
        save(outpath, fig)
    end
    return images
    
end

function give_image_rmsscatter(model::LensModel.ModelConfig, lens::Lenses.AbstractLens, param_ref::Dict{Tuple{Symbol,Symbol},Float64}, gridx::M, gridy::M) where M <: ROA
    """
    Computes the image positions based on the model configuration and lens parameters.
    Returns a scatter plot of the image positions.
    """
    adis = LensModel.LensModelUtils.adis_current(model, param_ref)

    sid = 1
    kid = 1
    sum_rms = 0.0
    count = 0
    for src in model.source_config.sources
        adis_value = adis[sid]
        kid = 1
        for knot in src.knots
            x = knot.x
            y = knot.y
            images_obs = [(xi, yi) for (xi, yi) in zip(x, y)]
            images_pred = predict_image(lens, gridx, gridy, x, y, adis_value, sid, kid, images_obs)
            println(length(images_obs), " Observed images: ", images_obs)
            println(length(images_pred), " Predicted images: ", images_pred)
            println("------------------------------------------------------")
            sum_rms += give_sum_rms(images_pred, images_obs)
            count += length(images_obs)
            kid += 1
        end
        sid += 1
    end
    
    return sqrt(sum_rms / count)
end

function give_sum_rms(images_pred, images_obs)
    """
    Computes rms between predicted and observed image positions. Can handle false predictions as well
    since it matches the pairs first. The arguments are vectors of tuples of (x, y) positions.
    """

    sum_rms = 0.0

    for image in images_obs
        pred_closest = nothing
        len_closest = Inf
        for pred in images_pred
            dist = sqrt((image[1] - pred[1])^2 + (image[2] - pred[2])^2)
            if dist < len_closest
                len_closest = dist
                pred_closest = pred
            end
        end
        sum_rms += len_closest^2
    end
    return sum_rms
end

function give_inversehessian(κ::M) where M <: ROA
    """
    Returns the covariance matrix (inverse Hessian) of the convergence map κ. This assumes a 
    gaussian distribution near the minima of the target function.
    """
    global prior_kappa, gridx, gridy, model, param_ref
    κ_vec = vec(κ)
    hessian = zeros(length(κ_vec), length(κ_vec))

    f0 = neg_logpost_MEM(κ_vec)
    h = 1e-5

    Threads.@threads for i in eachindex(κ_vec)
        buf = copy(κ_vec)
        buf[i] += h
        f1 = neg_logpost_MEM(buf)

        for j in eachindex(κ_vec)
            buf2 = copy(buf)
            buf2[j] += h
            f2 = neg_logpost_MEM(buf2)

            buf3 = copy(κ_vec)
            buf3[j] += h
            f3 = neg_logpost_MEM(buf3)

            hessian[i, j] = (f2 - f1 - f3 + f0) / (h^2)
            println("Hessian computation progress: ", i, ",", j)
        end
    end    
    return inv(hessian)
end

function give_errormap(hessian::M) where M <: ROA
    """
    Computes the error map from the inverse Hessian matrix.
    """
    global gridx
    diag_elements = diag(hessian)
    errormap = reshape(sqrt.(diag_elements), size(gridx))
    return errormap
end

function load_fitsfile(filename::String)
    """
    Loads a FITS file and returns coordinate grid along with kappa, gamma1, gamma2 matrices.
    """
    local gridx_fits, gridy_fits, kappa, gamma1, gamma2

    FITS("../$(filename)_data/kappa_z9_0.fits") do f
        hdr = read_header(f[1])
        kappa = read(f[1])

        NX = hdr["NAXIS1"]
        NY = hdr["NAXIS2"]

        pixel_scale = hdr["CDELT1"] * 3600.0  # Convert degrees to arcseconds
        CRVAL1 = hdr["CRVAL1"] * 3600.0
        CRVAL2 = hdr["CRVAL2"] * 3600.0
        CRPIX1 = hdr["CRPIX1"]
        CRPIX2 = hdr["CRPIX2"]

        X_max = (NX - CRPIX1) * pixel_scale + CRVAL1
        Y_max = (NY - CRPIX2) * pixel_scale + CRVAL2

        gridx_fits, gridy_fits = Lenses.get_meshgrid(X_max, Y_max, pixel_scale)
    end

    FITS("../$(filename)_data/gammax_z9_0.fits") do f
        gamma1 = read(f[1])
    end

    FITS("../$(filename)_data/gammay_z9_0.fits") do f
        gamma2 = read(f[1])
    end

    return gridx_fits[2:end, 2:end], gridy_fits[2:end, 2:end], kappa, gamma1, gamma2    # because the fits arrays are 2048, but grid_fits are 2049
end

load_fitsfile (generic function with 1 method)

In [20]:
X_lim = 125.0
Y_lim = 125.0
res = 2.5
gridx_fits, gridy_fits, kappa, gamma1, gamma2 = load_fitsfile("Ares")
kappa = Float64.(kappa)  # Ensure kappa is of type Float64
gamma1 = Float64.(gamma1)  # Ensure gamma1 is of type Float64
gamma2 = Float64.(gamma2)  # Ensure gamma2 is of type Float64
println(size(kappa))
println("gridx_fits size: ", size(gridx_fits))
println("gridy_fits siz e: ", size(gridy_fits))
println(gridx_fits[1,1])
println(gridy_fits[1,1])
println(gridx_fits[end,1])
println(gridy_fits[end,1])

kappa_finefits, gridx_finefits, gridy_finefits = refine_map(kappa, gridx_fits, gridy_fits, X_lim, Y_lim, res)
gamma1_finefits, _, _ = refine_map(gamma1, gridx_fits, gridy_fits, X_lim, Y_lim, res)
gamma2_finefits, _, _ = refine_map(gamma2, gridx_fits, gridy_fits, X_lim, Y_lim, res)
"""mag_finefits = zeros(size(kappa_finefits)) 
@. mag_finefits = 1.0 / ((1.0 - kappa_finefits)^2 - (gamma1_finefits^2 + gamma2_finefits^2))

tol = 1e-12
println("zeros     = ", count(x -> abs(x) < tol, mag_finefits))
println("negative  = ", count(<(0.0), mag_finefits))
println("positive  = ", count(>(0.0), mag_finefits))
println("NaNs      = ", count(isnan, mag_finefits))
println("Infs      = ", count(isinf, mag_finefits))
println("extrema   = ", extrema(mag_finefits))"""

(2048, 2048)
gridx_fits size: (2048, 2048)
gridy_fits siz e: (2048, 2048)
-149.926788
-149.926788
150.073344
-149.926788
before returning
(101, 101)
before returning
(101, 101)
before returning
(101, 101)


"mag_finefits = zeros(size(kappa_finefits)) \n@. mag_finefits = 1.0 / ((1.0 - kappa_finefits)^2 - (gamma1_finefits^2 + gamma2_finefits^2))\n\ntol = 1e-12\nprintln(\"zeros     = \", count(x -> abs(x) < tol, mag_finefits))\nprintln(\"negative  = \", count(<(0.0), mag_finefits))\nprintln(\"positive  = \", count(>(0.0), mag_finefits))\nprintln(\"NaNs      = \", count(isnan, mag_finefits))\nprintln(\"Infs      = \", count(isinf, mag_finefits))\nprintln(\"extrema   = \", extrema(mag_finefits))"

In [ ]:
fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], kappa_finefits, colormap = :turbo, colorrange = (0, 3.75))
cb = Colorbar(fig_[1,2], hm; label = "κ", width = 20)
save("../Ares_data/kappa_finefits.png", fig_)

fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], gamma1_finefits, colormap = :turbo)
cb = Colorbar(fig_[1,2], hm; label = "γ₁", width = 20)
save("../Ares_data/gamma1_finefits.png", fig_)

fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], gamma2_finefits, colormap = :turbo)
cb = Colorbar(fig_[1,2], hm; label = "γ₂", width = 20)
save("../Ares_data/gamma2_finefits.png", fig_)

fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], abs.(mag_finefits), colormap = :turbo, colorrange = (0, 100))
cb = Colorbar(fig_[1,2], hm; label = "|μ|", width = 20)
save("../Ares_data/mag_finefits.png", fig_)

In [32]:
name = "MEM_fit_result16x16_1_125FOV_reg1"
filename = "../Diagnostics/files/$name" * ".jld2"

global prior_kappa, gridx, gridy, model, param_ref, reg_factor
# loading the jld2 file
data = load(filename)
model = data["model_config"]
κ_map = data["κ_map"]
#prior_kappa = data["prior_kappa"]
reg_factor = data["reg_factor"]
param_ref = Dict(p.key => p.refer for p in model.parameters)

# making the grid
gridx, gridy = make_gridfrom_model(model)

κ_fine, x_fine, y_fine = refine_map(κ_map, gridx, gridy, X_lim, Y_lim, res)
# initialize the lens
free_lens = FreeFormLens.init_FreeFormLens(κ_fine, x_fine, y_fine)

println("Lens initialized.")

before returning
(101, 101)
Lens initialized.


In [33]:
zs = 9
zd = model.observation.z_d
cosmo = Cosmology.init_cosmology()      # default cosmo
Dds = Cosmology.angular_diameter_distance(cosmo, zd, zs)
Ds = Cosmology.angular_diameter_distance(cosmo, 0.0, zs)
adis = Dds / Ds

κ_map .*= adis
κ_fine .*= adis

ares_lens = FreeFormLens.init_FreeFormLens(kappa_finefits ./ adis, gridx_finefits, gridy_finefits)

Main.FreeFormLens.init_FreeFormLens(:FreeFormLens, [0.41816614426576415 0.41426713817375377 … 0.06275485859759686 0.05649326736141568; 0.42961397431477444 0.4327114909111675 … 0.06444592264812392 0.05900913097974889; … ; 0.09049412248148485 0.09158244877252299 … 0.2609306254556216 0.2694398371348305; 0.08567879174947665 0.08919863996062886 … 0.2467626072563861 0.2540527563632665], [-125.0 -125.0 … -125.0 -125.0; -122.5 -122.5 … -122.5 -122.5; … ; 122.5 122.5 … 122.5 122.5; 125.0 125.0 … 125.0 125.0], [-125.0 -122.5 … 122.5 125.0; -125.0 -122.5 … 122.5 125.0; … ; -125.0 -122.5 … 122.5 125.0; -125.0 -122.5 … 122.5 125.0])

In [34]:
scat = give_image_rmsscatter(model, ares_lens, param_ref, x_fine, y_fine)

3 Observed images: [(-10.7249, -50.577), (-30.9789, -42.4754), (-38.2703, -28.587)]
3 Predicted images: Tuple{Union{Float64, Int64}, Union{Float64, Int64}}[(-38.04533523694849, -28.76774001818502), (-31.262619299027886, -42.21447409276789), (-11.211867753967907, -50.370894717321406)]
------------------------------------------------------
3 Observed images: [(-27.1467, -38.5275), (-14.1842, -23.6232), (-33.898, -19.2252)]
5 Predicted images: Tuple{Union{Float64, Int64}, Union{Float64, Int64}}[(-33.58582956240813, -19.31702264453858), (-14.124686053165714, -23.905442781250485), (-20.291006380156595, -31.822987771461502), (-27.05252049487763, -38.63188312373149), (-3.4920841219272365, -45.37515008962147)]
------------------------------------------------------
4 Observed images: [(-2.2609, -48.8791), (-28.5952, -42.9699), (-12.8589, -24.3561), (-39.5144, -17.5605)]
5 Predicted images: Tuple{Union{Float64, Int64}, Union{Float64, Int64}}[(-39.07939385720599, -17.469611982342226), (-12.610205

InterruptException: InterruptException:

In [6]:
mag_fine = Lenses.get_magnification_image(free_lens, x_fine, y_fine, adis)

fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], (mag_fine .- mag_finefits)./mag_finefits, colormap = :BrBG, colorrange = (-1.0, 4.0))
cb = Colorbar(fig_[1,2], hm; label = L"(μ - μ_{truth})/μ_{truth}", width = 20)
save("../Diagnostics/plots/$(name)_mag_rel_deviation.png", fig_)

InterruptException: Error trying to display an error.

In [15]:
fig_, axes_ = Lenses.plot_sky(gridx_finefits, gridy_finefits)
hm = heatmap!(axes_, gridx_finefits[:,1], gridy_finefits[1,:], (κ_fine .- kappa_finefits)./kappa_finefits, colormap = :afmhot, colorrange = (-1.0, 2.0))
cb = Colorbar(fig_[1,2], hm; label = L"(κ - κ_{truth})/κ_{truth}", width = 20)
save("../Diagnostics/plots/$(name)_kappa_rel_deviation.png", fig_)

In [ ]:
errors .*= adis            # rescaling errors to source redshift 9

err_fig, err_axes = Lenses.plot_sky(gridx, gridy)
hm = heatmap!(err_axes, gridx[:,1], gridy[1,:], errors, colormap = :turbo, colorrange = (0, maximum(errors)))
cb = Colorbar(err_fig[1,2], hm; label = "δκ", width = 20)
save("../Diagnostics/plots/$name" * "_error_map.png", err_fig)

rel_errors = errors ./ κ_map
relerr_fig, relerr_axes = Lenses.plot_sky(gridx, gridy)
hm_rel = heatmap!(relerr_axes, gridx[:,1], gridy[1,:], rel_errors, colormap = :turbo, colorrange = (0, 2))
cb_rel = Colorbar(relerr_fig[1,2], hm_rel; label = "δκ/κ", width = 20)
save("../Diagnostics/plots/$name" * "_relative_error_map.png", relerr_fig)


In [ ]:
μ_fig, μ_axes = Lenses.plot_magnification_map(free_lens, x_fine, y_fine, adis, heatmap_kws = (colormap = :turbo, colorrange = (0,100)))
save("../Diagnostics/plots/$name" * "_magnification_map.png", μ_fig)

In [ ]:
cc_fig, cc_axes = Lenses.plot_image_plane(free_lens, x_fine, y_fine, adis, two_panel = true)
save("../Diagnostics/plots/$name" * "_critical_curves.png", cc_fig)

In [19]:
κ_fig, κ_axes = Lenses.plot_surface_density(free_lens, x_fine, y_fine, adis, unit = :convergence, heatmap_kws = (colormap = :turbo, colorrange = (0, 3.75)))
save("../Diagnostics/plots/$name" * "_kappa_map.png", κ_fig)

In [17]:
prof_fig, prof_axis = Lenses.plot_magnification_profile(free_lens, x_fine, y_fine, adis)
save("../Diagnostics/plots/$name" * "_magnification_profile.png", prof_fig)

In [ ]:
println("χ² of predicted image positions: ", data["chi2"])

InterruptException: InterruptException: